In [ ]:

import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT/".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False

con = duckdb.connect(str(DB_PATH), read_only= True)


In [ ]:
# 商品热度TOP 20
top_items = con.sql("""
SELECT 
    item_id,
    COUNT(*) AS pv,
    SUM(IF(behavior_type = '购买', 1, 0)) AS buy_cnt
FROM user_behavior
GROUP BY item_id
ORDER BY pv DESC
LIMIT 20
""").df()

fig, ax = plt.subplots(figsize = (10,7))
sns.barplot(data = top_items,y="pv",x = "item_id", ax=ax,color = "#4C72B0")
ax.set_title("商品 PV TOP20")
ax.set_xlabel("商品ID")
ax.tick_params(axis="x",rotation=45)
ax.set_ylabel("PV")
ax.grid(visible=True,axis="y",alpha=0.3)
apply_chinese(ax, font_prop)

plt.savefig(FIGURES_DIR/"top_items.png")
plt.show()

In [ ]:
# 品类分布
category = con.sql("""
SELECT 
    item_category,
    COUNT(*) AS actions,
    SUM(IF(behavior_type = '购买',1,0)) AS buy_cnt
FROM user_behavior
GROUP BY item_category
ORDER BY actions DESC
LIMIT 15
""").df()

fig,ax = plt.subplots(figsize=(10,5))
sns.barplot(data = category, x = "item_category",y="actions",ax=ax,color="#55A868")
ax.set_title("品类行为量 TOP 15")
ax.tick_params(axis="x", rotation=45)

apply_chinese(ax,font_prop)
plt.savefig(FIGURES_DIR/"category_dist.png")
plt.show()

In [ ]:
# 品类转化率
category["buy_rate"] = round(category["buy_cnt"]/category["actions"] * 100,2)

fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data = category,x="item_category",y="buy_rate",ax=ax,color="#C44E52")
ax.set_title("品类购买转化率")
ax.tick_params(axis="x",rotation=45)
apply_chinese(ax, font_prop)

plt.savefig(FIGURES_DIR/"category_rate.png")
plt.show()
